# Can Financial Ratios Predict ESG Scores?
## A Machine Learning Analysis of UK Listed Companies

**Author:** Md Aminul Islam Tushar
**Module:** BUSI 1783 — MSc Business Analytics, University of Greenwich
**Data:** FTSE 250 non-financial constituents, 2019–2023 (ORBIS + Refinitiv Eikon)

---

### Research questions

**RQ1** — To what extent can financial ratios predict ESG scores among UK-listed companies?
**RQ2** — Do machine learning models provide better predictive performance than traditional regression?

### Hypotheses

| | Statement |
|---|---|
| H1 | Profitability is positively associated with ESG performance |
| H2 | Leverage is negatively associated with ESG performance |
| H3 | Liquidity is positively associated with ESG performance |
| H4 | Random Forest and XGBoost outperform Multiple Linear Regression |

### A note on evaluation design

ESG scores are highly persistent within firms. A conventional random train/test split would
place the same company on both sides of the partition, allowing tree-based models to recognise
firms rather than learn the financial→ESG relationship. **All splits and cross-validation folds in
this notebook are grouped by firm (ISIN)**, so no company appears in both training and test data.
Section 7 quantifies how much this matters.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.model_selection import (GroupShuffleSplit, GroupKFold,
                                     RandomizedSearchCV, train_test_split,
                                     cross_val_score)
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

from xgboost import XGBRegressor
import shap

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 300,
                     'savefig.bbox': 'tight', 'font.size': 10})

from pathlib import Path
OUT = Path('outputs'); OUT.mkdir(exist_ok=True)

print('Libraries loaded.')

---
## 1. Load and prepare data

In [ ]:
df = pd.read_csv('data/final/analytical_dataset.csv')

print(f'Observations : {len(df)}')
print(f'Firms        : {df["ISIN"].nunique()}')
print(f'Years        : {df["Year"].min()}–{df["Year"].max()}')
df.head()

### 1.1 Collapse sparse sector categories

Sections with fewer than 10 firm-year observations cannot support reliable dummy estimation
and would produce empty cells in some cross-validation folds. These are merged into 'Other'.

In [ ]:
counts = df['SectorSection'].value_counts()
rare = counts[counts < 10].index.tolist()
df['SectorGrouped'] = df['SectorSection'].replace({s: 'Other' for s in rare})

print(f'Collapsed into "Other": {rare}')
print()
print(df['SectorGrouped'].value_counts().to_string())

In [ ]:
# Median-impute the small number of missing firm ages
n_missing = df['FirmAge'].isna().sum()
df['FirmAge'] = df['FirmAge'].fillna(df['FirmAge'].median())
print(f'FirmAge values imputed at median: {n_missing}')

---
## 2. Descriptive statistics — **Table 4.1**

In [ ]:
RATIOS   = ['ROA', 'ROE', 'DebtToEquity', 'DebtToAssets', 'CurrentRatio', 'QuickRatio']
CONTROLS = ['LogTotalAssets', 'FirmAge']
TARGET   = 'ESG_Score'

desc = df[RATIOS + CONTROLS + [TARGET]].describe().T
desc = desc[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']].round(3)
desc.columns = ['N', 'Mean', 'SD', 'Min', 'Q1', 'Median', 'Q3', 'Max']
desc.to_csv(OUT / 'table_4_1_descriptives.csv')
desc

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))

axes[0].hist(df[TARGET], bins=25, color='#4C72B0', edgecolor='white')
axes[0].set_title('Distribution of ESG scores'); axes[0].set_xlabel('ESG score')

yearly = df.groupby('Year')[TARGET].mean()
axes[1].plot(yearly.index, yearly.values, marker='o', color='#4C72B0')
axes[1].set_title('Mean ESG score by year'); axes[1].set_xlabel('Year')

order = df.groupby('SectorGrouped')[TARGET].median().sort_values().index
sns.boxplot(data=df, y='SectorGrouped', x=TARGET, order=order, ax=axes[2], color='#4C72B0')
axes[2].set_title('ESG score by sector'); axes[2].set_ylabel('')

plt.tight_layout()
plt.savefig(OUT / 'figure_4_1_distributions.png')
plt.show()

---
## 3. Correlation analysis and multicollinearity — **Table 4.2**

In [ ]:
corr = df[RATIOS + CONTROLS + [TARGET]].corr()

plt.figure(figsize=(8, 6.5))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True, cbar_kws={'shrink': .8})
plt.title('Pearson correlation matrix')
plt.tight_layout()
plt.savefig(OUT / 'figure_4_2_correlation.png')
plt.show()

corr.round(3).to_csv(OUT / 'table_4_2_correlations.csv')

In [ ]:
# Bivariate correlations with the dependent variable
biv = corr[TARGET].drop(TARGET).sort_values()
print('Correlation with ESG score:')
print(biv.round(3).to_string())

In [ ]:
X_vif = sm.add_constant(df[RATIOS + CONTROLS])
vif = pd.DataFrame({
    'Variable': X_vif.columns,
    'VIF': [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
})
vif = vif[vif['Variable'] != 'const'].round(2)
vif.to_csv(OUT / 'table_4_3_vif.csv', index=False)

print('Variance inflation factors (values above 10 indicate a problem):')
print(vif.to_string(index=False))

---
## 4. Build the model matrix

In [ ]:
sector_dummies = pd.get_dummies(df['SectorGrouped'], prefix='Sec', drop_first=True).astype(float)
year_dummies   = pd.get_dummies(df['Year'], prefix='Yr', drop_first=True).astype(float)

X = pd.concat([df[RATIOS + CONTROLS], sector_dummies, year_dummies], axis=1)
y = df[TARGET]
groups = df['ISIN']

# Standardise continuous predictors only; dummies remain 0/1
scaler = StandardScaler()
X_scaled = X.copy()
X_scaled[RATIOS + CONTROLS] = scaler.fit_transform(X[RATIOS + CONTROLS])

print(f'Design matrix: {X_scaled.shape[0]} observations × {X_scaled.shape[1]} features')
print(f'  {len(RATIOS)} ratios, {len(CONTROLS)} controls, '
      f'{sector_dummies.shape[1]} sector dummies, {year_dummies.shape[1]} year dummies')

### 4.1 Grouped train/test split

The split is stratified by firm, not by observation. Every year of a given company falls
entirely on one side of the partition.

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X_scaled, y, groups=groups))

X_train, X_test = X_scaled.iloc[train_idx], X_scaled.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
g_train = groups.iloc[train_idx]

assert set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]) == set(), 'Firm overlap detected'

print(f'Train : {len(train_idx):>3} obs, {g_train.nunique():>2} firms')
print(f'Test  : {len(test_idx):>3} obs, {groups.iloc[test_idx].nunique():>2} firms')
print('No firm appears in both partitions.')

In [ ]:
def evaluate(name, y_true, y_pred):
    return {
        'Model': name,
        'R²':    r2_score(y_true, y_pred),
        'RMSE':  np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAE':   mean_absolute_error(y_true, y_pred),
    }

results = []

---
## 5. Model 1 — Multiple Linear Regression (baseline)

Fitted with `statsmodels` to obtain coefficients, standard errors, and p-values, which
address H1–H3 directly.

In [ ]:
ols = sm.OLS(y_train, sm.add_constant(X_train)).fit()
mlr_pred = ols.predict(sm.add_constant(X_test))
results.append(evaluate('Multiple Linear Regression', y_test, mlr_pred))

print(ols.summary().tables[0])

In [ ]:
def stars(p):
    return '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.10 else ''

coef_table = pd.DataFrame({
    'Coefficient': ols.params,
    'Std. Error':  ols.bse,
    't':           ols.tvalues,
    'p-value':     ols.pvalues,
}).loc[RATIOS + CONTROLS].round(4)
coef_table['Sig.'] = coef_table['p-value'].apply(stars)
coef_table.to_csv(OUT / 'table_4_4_mlr_coefficients.csv')

print('MLR coefficients on standardised predictors')
print('(a one-SD increase in the predictor changes the ESG score by the coefficient)')
print()
print(coef_table.to_string())
print()
print('*** p<0.01, ** p<0.05, * p<0.10')

---
## 6. Models 2 and 3 — Random Forest and XGBoost

Hyperparameters are tuned by randomised search within **grouped** cross-validation folds
drawn from the training partition only, so no test-partition information influences model
selection.

In [ ]:
gkf_tune = GroupKFold(n_splits=5)

rf_grid = {
    'n_estimators':     [300, 500],
    'max_depth':        [3, 5, 8, None],
    'min_samples_leaf': [1, 3, 5, 10],
    'max_features':     ['sqrt', 0.5, 1.0],
}

rf_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    rf_grid, n_iter=20, scoring='r2',
    cv=list(gkf_tune.split(X_train, y_train, groups=g_train)),
    random_state=RANDOM_STATE, n_jobs=-1)

rf_search.fit(X_train, y_train)
rf = rf_search.best_estimator_

print('Random Forest — best parameters:')
for k, v in rf_search.best_params_.items():
    print(f'  {k:<18} {v}')
print(f'  grouped CV R²      {rf_search.best_score_:.4f}')

In [ ]:
xgb_grid = {
    'n_estimators':     [200, 500],
    'learning_rate':    [0.01, 0.05, 0.10],
    'max_depth':        [2, 3, 4, 6],
    'subsample':        [0.7, 0.9],
    'colsample_bytree': [0.7, 0.9],
    'min_child_weight': [1, 5, 10],
}

xgb_search = RandomizedSearchCV(
    XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    xgb_grid, n_iter=25, scoring='r2',
    cv=list(gkf_tune.split(X_train, y_train, groups=g_train)),
    random_state=RANDOM_STATE, n_jobs=-1)

xgb_search.fit(X_train, y_train)
xgb = xgb_search.best_estimator_

print('XGBoost — best parameters:')
for k, v in xgb_search.best_params_.items():
    print(f'  {k:<18} {v}')
print(f'  grouped CV R²      {xgb_search.best_score_:.4f}')

In [ ]:
results.append(evaluate('Random Forest', y_test, rf.predict(X_test)))
results.append(evaluate('XGBoost',       y_test, xgb.predict(X_test)))

comparison = pd.DataFrame(results).round(4)
comparison.to_csv(OUT / 'table_4_5_model_comparison.csv', index=False)

print('Held-out test performance (grouped split)')
print(comparison.to_string(index=False))

---
## 7. Why the split design matters

This section demonstrates the consequence of ignoring the panel structure. The identical
models are re-fitted using a conventional random split, in which the same firm may appear
in both training and test data.

This is presented as a methodological diagnostic, not as a result.

In [ ]:
Xtr_n, Xte_n, ytr_n, yte_n = train_test_split(
    X_scaled, y, test_size=0.2, random_state=RANDOM_STATE)

naive = []
for name, model in [('Multiple Linear Regression', LinearRegression()),
                    ('Random Forest', RandomForestRegressor(**rf_search.best_params_,
                                                            random_state=RANDOM_STATE, n_jobs=-1)),
                    ('XGBoost', XGBRegressor(**xgb_search.best_params_,
                                             random_state=RANDOM_STATE, n_jobs=-1))]:
    model.fit(Xtr_n, ytr_n)
    naive.append(evaluate(name, yte_n, model.predict(Xte_n)))

naive_df = pd.DataFrame(naive).round(4)

side_by_side = comparison[['Model', 'R²']].merge(
    naive_df[['Model', 'R²']], on='Model', suffixes=(' (grouped)', ' (random)'))
side_by_side['Inflation'] = (side_by_side['R² (random)'] - side_by_side['R² (grouped)']).round(4)
side_by_side.to_csv(OUT / 'table_4_6_leakage_diagnostic.csv', index=False)

print('Effect of ignoring panel structure')
print(side_by_side.to_string(index=False))

In [ ]:
within  = df.groupby('ISIN')[TARGET].std().mean()
between = df.groupby('ISIN')[TARGET].mean().std()

print(f'Mean within-firm SD of ESG score : {within:.2f}')
print(f'Between-firm SD of ESG score     : {between:.2f}')
print(f'Ratio                            : {between/within:.1f}×')
print()
print('ESG scores vary far more across firms than within them over time.')
print('Under a random split, tree models can identify the firm from its financial')
print('profile and reproduce a score already seen in training.')

---
## 8. Cross-validated performance — **Table 4.7**

Ten-fold grouped cross-validation across the full sample, providing a more stable estimate
than a single train/test split.

In [ ]:
gkf10 = GroupKFold(n_splits=10)
folds = list(gkf10.split(X_scaled, y, groups=groups))

cv_rows = []
for name, model in [('Multiple Linear Regression', LinearRegression()),
                    ('Random Forest', RandomForestRegressor(**rf_search.best_params_,
                                                            random_state=RANDOM_STATE, n_jobs=-1)),
                    ('XGBoost', XGBRegressor(**xgb_search.best_params_,
                                             random_state=RANDOM_STATE, n_jobs=-1))]:
    r2  = cross_val_score(model, X_scaled, y, cv=folds, scoring='r2')
    mae = -cross_val_score(model, X_scaled, y, cv=folds, scoring='neg_mean_absolute_error')
    cv_rows.append({'Model': name,
                    'CV R² (mean)': r2.mean(), 'CV R² (SD)': r2.std(),
                    'CV MAE (mean)': mae.mean()})

cv_df = pd.DataFrame(cv_rows).round(4)
cv_df.to_csv(OUT / 'table_4_7_cross_validation.csv', index=False)
print('Ten-fold grouped cross-validation')
print(cv_df.to_string(index=False))

---
## 9. SHAP interpretability — **Figure 4.3**

SHAP values are computed for the better-performing ensemble model, decomposing each
prediction into the marginal contribution of every feature.

In [ ]:
rf_r2  = comparison.loc[comparison.Model == 'Random Forest', 'R²'].iloc[0]
xgb_r2 = comparison.loc[comparison.Model == 'XGBoost', 'R²'].iloc[0]

best_model, best_name = (rf, 'Random Forest') if rf_r2 >= xgb_r2 else (xgb, 'XGBoost')
print(f'Computing SHAP values for: {best_name}')

explainer   = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test)

In [ ]:
plt.figure()
shap.summary_plot(shap_values, X_test, plot_type='bar', show=False, max_display=15)
plt.title(f'Mean absolute SHAP value — {best_name}')
plt.tight_layout()
plt.savefig(OUT / 'figure_4_3_shap_importance.png')
plt.show()

In [ ]:
plt.figure()
shap.summary_plot(shap_values, X_test, show=False, max_display=15)
plt.title(f'SHAP value distribution — {best_name}')
plt.tight_layout()
plt.savefig(OUT / 'figure_4_4_shap_beeswarm.png')
plt.show()

In [ ]:
shap_imp = pd.DataFrame({
    'Feature': X_test.columns,
    'Mean |SHAP|': np.abs(shap_values).mean(axis=0)
}).sort_values('Mean |SHAP|', ascending=False).round(3)

shap_imp.to_csv(OUT / 'table_4_8_shap_importance.csv', index=False)
print(shap_imp.head(12).to_string(index=False))

---
## 10. Hypothesis verdicts — **Table 4.9**

H1–H3 are assessed on the sign and statistical significance of the MLR coefficients, which
are estimated with controls for firm size, sector, and year. H4 is assessed by comparing
held-out and cross-validated performance across the three models.

In [ ]:
def verdict(variables, expected_sign):
    lines = []
    for v in variables:
        b = coef_table.loc[v, 'Coefficient']
        p = coef_table.loc[v, 'p-value']
        sign_ok = (b > 0) if expected_sign == '+' else (b < 0)
        if p >= 0.10:
            outcome = 'not significant'
        else:
            outcome = 'supported' if sign_ok else 'significant, opposite sign'
        lines.append(f'{v}: β={b:+.3f}, p={p:.3f} → {outcome}')
    return lines

print('H1 — Profitability positively associated with ESG')
for l in verdict(['ROA', 'ROE'], '+'): print('   ', l)
print()
print('H2 — Leverage negatively associated with ESG')
for l in verdict(['DebtToEquity', 'DebtToAssets'], '-'): print('   ', l)
print()
print('H3 — Liquidity positively associated with ESG')
for l in verdict(['CurrentRatio', 'QuickRatio'], '+'): print('   ', l)
print()
print('H4 — Ensemble methods outperform MLR')
best_row = comparison.loc[comparison['R²'].idxmax()]
print(f"    Best held-out R²: {best_row['Model']} ({best_row['R²']:.4f})")
print(f"    Best CV R²      : {cv_df.loc[cv_df['CV R² (mean)'].idxmax(), 'Model']}")

In [ ]:
# Assemble the verdict table for Chapter 4
rows = []
for h, vars_, sign, statement in [
    ('H1', ['ROA', 'ROE'], '+', 'Profitability positively associated with ESG'),
    ('H2', ['DebtToEquity', 'DebtToAssets'], '-', 'Leverage negatively associated with ESG'),
    ('H3', ['CurrentRatio', 'QuickRatio'], '+', 'Liquidity positively associated with ESG'),
]:
    sig = [v for v in vars_ if coef_table.loc[v, 'p-value'] < 0.10]
    correct = [v for v in sig if (coef_table.loc[v, 'Coefficient'] > 0) == (sign == '+')]
    if not sig:
        outcome = 'Not supported'
    elif len(correct) == len(sig):
        outcome = 'Supported'
    elif correct:
        outcome = 'Mixed'
    else:
        outcome = 'Contradicted'
    detail = '; '.join(f"{v} β={coef_table.loc[v,'Coefficient']:+.2f}{stars(coef_table.loc[v,'p-value'])}"
                       for v in vars_)
    rows.append({'Hypothesis': h, 'Statement': statement, 'Evidence': detail, 'Verdict': outcome})

h4_supported = max(comparison.loc[comparison.Model != 'Multiple Linear Regression', 'R²']) > \
               comparison.loc[comparison.Model == 'Multiple Linear Regression', 'R²'].iloc[0]
rows.append({'Hypothesis': 'H4',
             'Statement': 'Ensemble models outperform MLR',
             'Evidence': '; '.join(f"{r['Model']} R²={r['R²']:.3f}" for _, r in comparison.iterrows()),
             'Verdict': 'Supported' if h4_supported else 'Not supported'})

verdicts = pd.DataFrame(rows)
verdicts.to_csv(OUT / 'table_4_9_hypothesis_verdicts.csv', index=False)
verdicts

---
## 11. Summary of exported files

All tables and figures referenced in Chapter 4 are written to `outputs/`.

In [ ]:
for f in sorted(OUT.iterdir()):
    print(f'  {f.name}')